In [166]:
import sys, glob, os, yaml, sparse, tracemalloc, vcf
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy.stats as st
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from dna_features_viewer import BiopythonTranslator, GraphicFeature, GraphicRecord
from dna_features_viewer.biotools import annotate_biopython_record

import warnings
warnings.filterwarnings("ignore")

who_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/WHO_catalog_clean.csv")

sys.path.append(os.path.join(os.getcwd(), "utils_files"))
from model_utils import *
from data_utils import *
from dataloader import MtbGeneDataset

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
print(f"Tensorflow version: {tf.__version__}")

# to use this, add @tf.function before the loss function (not the wrapper, the inner loop). Don't do this though, keep the default eager execution because you are not an expert haha
# from tensorflow.python.framework.ops import disable_eager_ebxecution
# disable_eager_execution()

Tensorflow version: 2.11.0


In [202]:
rif_rpoBC_df = pd.read_csv("analysis/Rifampicin/rpoBC_saliency_results.csv")

# when saving, the Pos column gets converted to strings because it contains a mixture of strings and floats
for i, row in rif_rpoBC_df.iterrows():
    str_num = row["Pos"].replace(".0", "")
    if str_num.isnumeric():
        rif_rpoBC_df.loc[i, "Pos"] = str(int(str_num))